# E018 — Audio VTLP augmentation

Adds Vocal Tract Length Perturbation (VTLP) to the E008 +All baseline.
VTLP warps the spectrogram frequency axis by factor α, simulating biological
variation in vocal tract length — a different axis than speed perturbation.

**Configs tested:**
1. **E008 +All** (reference): original + noise20 + speed (3 copies)
2. **+All+VTLP**: original + noise20 + speed + vtlp (4 copies)
3. **+VTLP_replace_speed**: original + noise20 + vtlp (3 copies)

E008 +All reference numbers: fold0=3.47, fold1=8.33, fold2=0.83, mean=4.21±3.11%

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve, auc
from scipy.special import logsumexp
from scipy.interpolate import interp1d
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
CONFIG_COLORS = {
    "E008 +All":           "#95A5A6",
    "+All+VTLP":           "#E74C3C",
    "+VTLP_replace_speed": "#8E44AD",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Augmentation functions

All augmentations operate on the raw waveform `y` (float32 array).
MFCC extraction and CMN happen **after** augmentation.

VTLP warps the frequency axis in the STFT domain:
- `α > 1`: stretches low frequencies, compresses high → larger apparent vocal tract
- `α < 1`: compresses low frequencies, stretches high → smaller apparent vocal tract

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def aug_noise(y: np.ndarray, snr_db: float = 20.0, rng: np.random.Generator = None) -> np.ndarray:
    """Add white noise at target SNR."""
    signal_power = np.mean(y ** 2) + 1e-10
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(y)).astype(y.dtype)
    return y + noise


def aug_speed(y: np.ndarray, rate_range=(0.9, 1.1), rng: np.random.Generator = None) -> np.ndarray:
    """Random time stretch without changing pitch."""
    rate = rng.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)


def aug_vtlp(y: np.ndarray, sr: int, rng: np.random.Generator, alpha_range=(0.9, 1.1)) -> np.ndarray:
    """Vocal Tract Length Perturbation via spectrogram frequency warping.

    Warps the frequency axis of the STFT by factor α, simulating biological
    variation in vocal tract length. α > 1 → formants shift up (shorter tract),
    α < 1 → formants shift down (longer tract). Different from speed perturbation
    which affects the time axis.
    """
    alpha = rng.uniform(*alpha_range)
    D = librosa.stft(y)                          # (n_fft//2+1, T) complex
    n_freq = D.shape[0]
    src_idx = np.arange(n_freq, dtype=float)
    dst_idx = np.clip(alpha * src_idx, 0, n_freq - 1)
    # Interpolate real and imaginary parts separately
    f_real = interp1d(src_idx, D.real, axis=0, kind='linear', fill_value='extrapolate')
    f_imag = interp1d(src_idx, D.imag, axis=0, kind='linear', fill_value='extrapolate')
    D_warped = f_real(dst_idx) + 1j * f_imag(dst_idx)
    y_warped = librosa.istft(D_warped, length=len(y))
    return y_warped.astype(y.dtype)


def extract_mfcc(y: np.ndarray, sr: int, n_mfcc: int = 13) -> np.ndarray:
    """MFCC + delta + delta-delta + CMN. Returns (T, 39)."""
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T
    mfcc  -= mfcc.mean(axis=0)  # CMN
    return mfcc


def load_and_augment(wav_path: Path, config: str, rng: np.random.Generator):
    """
    Load WAV and return list of (y, sr) tuples per config:
    - e008_all:            [original, noisy, speed-perturbed]
    - all_vtlp:            [original, noisy, speed-perturbed, vtlp-warped]
    - vtlp_replace_speed:  [original, noisy, vtlp-warped]
    """
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    samples = [(y, sr)]

    if config in ("e008_all", "all_vtlp", "vtlp_replace_speed"):
        samples.append((aug_noise(y, snr_db=20.0, rng=rng), sr))

    if config in ("e008_all", "all_vtlp"):
        samples.append((aug_speed(y, rng=rng), sr))

    if config in ("all_vtlp", "vtlp_replace_speed"):
        samples.append((aug_vtlp(y, sr, rng=rng), sr))

    return samples


def extract_batch(df: pd.DataFrame, data_dir: Path, config: str, seed: int):
    """Extract MFCC frames for all samples, with augmentation on train fold."""
    rng = np.random.default_rng(seed)
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in load_and_augment(find_wav(row["stem"], data_dir), config, rng):
            mfcc = extract_mfcc(y_aug, sr)
            all_mfcc.append(mfcc)
            all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


def score_utterance(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc  = extract_mfcc(y, sr)
    return float((adapted.score_samples(mfcc) - ubm.score_samples(mfcc)).mean())


print("Augmentation functions defined.")

## 2. VTLP visualization

Compare the spectrogram of the original vs VTLP-warped signal to confirm
the frequency shift is plausible (formants shift, duration unchanged).

In [ ]:
rng_viz = np.random.default_rng(42)
sample_row = manifest[manifest.label == 1].iloc[0]
y_orig, sr = librosa.load(find_wav(sample_row["stem"], DATA), sr=None, mono=True)

# Two VTLP copies with different alpha
rng_low = np.random.default_rng(10)
rng_high = np.random.default_rng(20)
# Force specific alphas for visualization
class FixedAlphaRng:
    def __init__(self, alpha):
        self._alpha = alpha
    def uniform(self, lo, hi):
        return self._alpha
    def normal(self, *a, **kw):
        return np.random.default_rng(0).normal(*a, **kw)

y_vtlp_low  = aug_vtlp(y_orig, sr, FixedAlphaRng(0.90))  # α=0.90, longer tract
y_vtlp_high = aug_vtlp(y_orig, sr, FixedAlphaRng(1.10))  # α=1.10, shorter tract
y_speed     = aug_speed(y_orig, rng=rng_viz)

signals = {
    "Original": y_orig,
    "VTLP α=0.90 (longer tract)": y_vtlp_low,
    "VTLP α=1.10 (shorter tract)": y_vtlp_high,
    "Speed ±10% (for contrast)": y_speed,
}

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
for row_idx, (title, y_sig) in enumerate(signals.items()):
    t = np.linspace(0, len(y_sig)/sr, len(y_sig))

    # Waveform
    ax = axes[row_idx, 0]
    ax.plot(t, y_sig, color=COLORS["nontarget"], lw=0.5, alpha=0.8)
    ax.set_title(f"{title} — waveform", fontsize=9)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Amplitude")

    # MFCC (static 13)
    ax = axes[row_idx, 1]
    mfcc_sig = extract_mfcc(y_sig, sr)[:, :13]
    im = ax.imshow(mfcc_sig.T, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(f"{title} — MFCC (static 13)", fontsize=9)
    ax.set_xlabel("Frame")
    ax.set_ylabel("MFCC coeff")

plt.suptitle(f"VTLP vs Speed perturbation — {sample_row['stem']} (target)",
             color=COLORS["target"], fontsize=12)
plt.tight_layout()
plt.show()
print("VTLP shifts MFCC pattern without changing duration (unlike speed).")

## 3. UBM + MAP functions (same as E008)

In [ ]:
def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    log_prob   = ubm._estimate_log_prob(X_target)
    log_resp   = log_prob + np.log(ubm.weights_)
    log_resp  -= logsumexp(log_resp, axis=1, keepdims=True)
    resp       = np.exp(log_resp)
    n_k        = resp.sum(axis=0)
    mu_hat     = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha      = n_k / (n_k + r)
    adapted    = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


print("UBM+MAP functions defined.")

## 4. Cross-validation — all three configs

Each config trains UBM and MAP on augmented train frames.
Val utterances are always scored from original WAVs (no augmentation leak).

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0

# E008 reference numbers (from notebook output, not recomputed to save time)
E008_REF = {
    "fold_eers": [3.47, 8.33, 0.83],
    "mean": 4.21,
    "std": 3.11,
    "min_dcf_mean": 0.0509,
}

CONFIGS = {
    "e008_all":           "E008 +All",
    "all_vtlp":           "+All+VTLP",
    "vtlp_replace_speed": "+VTLP_replace_speed",
}

all_results = {}
all_oof     = {}

for config_key, config_name in CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"Config: {config_name}")
    print('='*55)

    oof_scores   = np.full(len(manifest), np.nan)
    fold_results = []

    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        train_df = manifest.loc[train_idx]
        val_df   = manifest.loc[val_idx]

        # Augmented train frames only on train fold
        X_train, y_train = extract_batch(train_df, DATA, config_key, seed=SEED+fold_id)
        X_nt = X_train[y_train == 0]
        X_t  = X_train[y_train == 1]

        n_copies = {"e008_all": 3, "all_vtlp": 4, "vtlp_replace_speed": 3}[config_key]
        print(f"  Fold {fold_id}: {len(X_t)} target frames, {len(X_nt)} non-target frames ({n_copies}× aug)")

        ubm     = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
        adapted = map_adapt(ubm, X_t, r=MAP_R)

        # Score val on ORIGINAL WAVs only
        for idx, row in val_df.iterrows():
            oof_scores[idx] = score_utterance(find_wav(row["stem"], DATA), adapted, ubm)

        val_scores = oof_scores[val_idx]
        val_labels = manifest.loc[val_idx, "label"].to_numpy()
        eer, _     = compute_eer(val_scores[val_labels==1], val_scores[val_labels==0])
        min_dcf, _ = compute_min_dcf(val_scores[val_labels==1], val_scores[val_labels==0])
        fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
        print(f"  → EER={eer*100:.2f}%, min-DCF={min_dcf:.4f}")

    all_results[config_name] = fold_results
    all_oof[config_name]     = oof_scores.copy()

print("\nAll configs done.")

## 5. Results table

In [ ]:
print(f"{'Config':<25} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 77)

summary = []
for config_name, fold_results in all_results.items():
    eers   = [r["eer"]*100  for r in fold_results]
    dcfs   = [r["min_dcf"] for r in fold_results]
    mean_e = np.mean(eers)
    std_e  = np.std(eers)
    mean_d = np.mean(dcfs)
    marker = " ← E008 ref" if config_name == "E008 +All" else ""
    print(f"{config_name:<25} {eers[0]:>8.2f} {eers[1]:>8.2f} {eers[2]:>8.2f} "
          f"{mean_e:>8.2f} {std_e:>8.2f} {mean_d:>9.4f}{marker}")
    summary.append({"config": config_name, "f0": eers[0], "f1": eers[1], "f2": eers[2],
                    "mean": mean_e, "std": std_e, "min_dcf": mean_d})

print("-" * 77)
best = min(summary, key=lambda x: x["mean"])
print(f"\nBest config: {best['config']}  EER={best['mean']:.2f}±{best['std']:.2f}%")
print(f"E008 reference: EER=4.21±3.11%")
delta = best["mean"] - 4.21
direction = "improvement" if delta < 0 else "regression"
print(f"Delta vs E008: {delta:+.2f}% ({direction})")

oof_best = all_oof[best["config"]]
eer_oof, _   = compute_eer(oof_best[y_all==1], oof_best[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_best[y_all==1], oof_best[y_all==0])
print(f"OOF overall ({best['config']}): EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

## 6. Comparison bar chart

In [ ]:
configs = list(all_results.keys())
n_configs = len(configs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-fold grouped bars
ax = axes[0]
x = np.arange(3)
width = 0.25
for i, (config_name, fold_results) in enumerate(all_results.items()):
    eers   = [r["eer"]*100 for r in fold_results]
    offset = (i - n_configs/2 + 0.5) * width
    ax.bar(x + offset, eers, width, label=config_name,
           color=CONFIG_COLORS[config_name], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("Per-fold EER — VTLP augmentation")
ax.legend(fontsize=9)

# Mean ± std
ax = axes[1]
means  = [np.mean([r["eer"]*100 for r in fr]) for fr in all_results.values()]
stds   = [np.std( [r["eer"]*100 for r in fr]) for fr in all_results.values()]
colors_list = [CONFIG_COLORS[c] for c in configs]
bars = ax.bar(range(n_configs), means, color=colors_list, alpha=0.85,
              yerr=stds, capsize=6, error_kw=dict(elinewidth=2))
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.15,
            f"{m:.1f}\n±{s:.1f}", ha="center", fontsize=8)
best_idx = np.argmin(means)
bars[best_idx].set_edgecolor("gold")
bars[best_idx].set_linewidth(3)
ax.annotate("★ best", xy=(best_idx, means[best_idx] - stds[best_idx] - 0.3),
            ha="center", fontsize=9, color="goldenrod", fontweight="bold")
# Add E008 reference line
ax.axhline(4.21, color="black", ls="--", lw=1.5, alpha=0.7, label="E008 +All ref (4.21%)")
ax.set_xticks(range(n_configs))
ax.set_xticklabels(configs, fontsize=9)
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("Mean ± std — VTLP augmentation")
ax.legend(fontsize=8)

plt.suptitle("E018 — VTLP augmentation vs E008 +All baseline", fontsize=13)
plt.tight_layout()
plt.show()

## 7. DET curves

In [ ]:
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos    = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

fig, ax = plt.subplots(figsize=(7, 6))
for config_name, oof_s in all_oof.items():
    valid = ~np.isnan(oof_s)
    fpr, tpr, _ = roc_curve(y_all[valid], oof_s[valid])
    far_c = np.clip(fpr, 1e-4, 1-1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
    eer_c, _ = compute_eer(oof_s[valid & (y_all==1)], oof_s[valid & (y_all==0)])
    is_best = (config_name == best["config"])
    lw = 2.5 if is_best else 1.5
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=CONFIG_COLORS[config_name], lw=lw,
            label=f"{config_name}  EER={eer_c*100:.1f}%",
            zorder=5 if is_best else 1)

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curves — VTLP configs (OOF)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Summary and conclusion

In [ ]:
print("=" * 60)
print("E018 VTLP augmentation — final summary")
print("=" * 60)
print(f"{'Config':<25} {'Mean EER':>10} {'Std EER':>10} {'min-DCF':>10}")
print("-" * 60)
print(f"{'E008 +All (reference)':<25} {'4.21':>10} {'3.11':>10} {'0.0509':>10}")
for s in summary:
    if s["config"] != "E008 +All":
        print(f"{s['config']:<25} {s['mean']:>10.2f} {s['std']:>10.2f} {s['min_dcf']:>10.4f}")
print("-" * 60)
print(f"\nBest new config: {best['config']}")
delta = best["mean"] - 4.21
print(f"vs E008 +All:    {delta:+.2f}% ({'improvement' if delta < 0 else 'regression'})")
print("=" * 60)